# 🏷️ Logistic Regression — Solutions Notebook

**Complete, verified solutions.**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete! ✅')

### 3.1 Implement Sigmoid

In [ ]:
# ✅ SOLUTION: Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

assert sigmoid(0) == 0.5
assert np.isclose(sigmoid(6), 0.9975, atol=1e-3)
assert np.isclose(sigmoid(-6), 0.00247, atol=1e-3)
print('Sigmoid test passed! ✅')

### 3.2 Implement Cost Function & Gradient Descent

In [ ]:
# ✅ SOLUTION: Log-loss and Gradient Descent
def compute_cost(X, y, theta):
    m = len(y)
    h = sigmoid(X @ theta)
    eps = 1e-15  # Numerical stability
    cost = -(1 / m) * np.sum(y * np.log(h + eps) + (1 - y) * np.log(1 - h + eps))
    return cost

def fit_logistic_regression(X, y, alpha=0.1, num_iters=1000):
    m, n = X.shape
    theta = np.zeros((n, 1))
    y = y.reshape(-1, 1)
    history = []
    
    for i in range(num_iters):
        h = sigmoid(X @ theta)
        gradient = (1 / m) * X.T @ (h - y)
        theta = theta - alpha * gradient
        history.append(compute_cost(X, y, theta))
        
    return theta, history

### 3.3 Synthetic Classification & Plotting Decision Boundary

In [ ]:
X_raw, y_raw = make_blobs(n_samples=200, centers=2, n_features=2, random_state=42, cluster_std=1.5)
X_b = np.c_[np.ones((len(X_raw), 1)), X_raw]

theta_learned, cost_history = fit_logistic_regression(X_b, y_raw, alpha=0.1, num_iters=2000)

print(f'Learned Parameters:\n{theta_learned.ravel()}')
print(f'Final Cost: {cost_history[-1]:.4f}')

# Plot Decision Boundary: theta0 + theta1*x1 + theta2*x2 = 0 => x2 = -(theta0 + theta1*x1)/theta2
plt.figure(figsize=(10, 6))
plt.scatter(X_raw[:, 0], X_raw[:, 1], c=y_raw, cmap='bwr', alpha=0.8, edgecolors='k')

x1_vals = np.linspace(X_raw[:, 0].min() - 1, X_raw[:, 0].max() + 1, 100)
x2_vals = -(theta_learned[0] + theta_learned[1] * x1_vals) / theta_learned[2]
plt.plot(x1_vals, x2_vals, 'g--', linewidth=2, label='Decision Boundary')

plt.xlabel('X1')
plt.ylabel('X2')
plt.title('Logistic Regression Decision Boundary (From Scratch)')
plt.legend()
plt.show()

### 📦 Section 4: scikit-learn Comparison

In [ ]:
model = LogisticRegression()
model.fit(X_raw, y_raw)

y_pred = model.predict(X_raw)
y_prob = model.predict_proba(X_raw)[:, 1]

print('Accuracy: ', accuracy_score(y_raw, y_pred))
print('Precision:', precision_score(y_raw, y_pred))
print('Recall:   ', recall_score(y_raw, y_pred))
print('F1-Score: ', f1_score(y_raw, y_pred))
print('ROC-AUC:  ', roc_auc_score(y_raw, y_prob))

cm = confusion_matrix(y_raw, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()